In [1]:
!pip install torch --index-url https://download.pytorch.org/whl/cpu --quiet

In [2]:
!pip install pdfplumber 
!pip install nltk 
!pip install transformers==4.35.0 accelerate==0.24.0 huggingface_hub==0.17.3 tokenizers==0.14.1 --quiet
!pip install tokenizers 
!pip install sentencepiece protobuf --quiet
#Can use different tools apart from pdfplumber such as Camelot, PyMuPDF, etc.

In [3]:
import pdfplumber
from pathlib import Path

pdf_path = r"C:\Users\sarve\Downloads\Machine Learning in Artificial Intelligence.pdf"

pdf_name = Path(pdf_path).stem
output_text = f"{pdf_name}_extracted.txt"
output_result = f"{pdf_name}_results.txt"

with pdfplumber.open(pdf_path) as pdf:     #Opens the PDF File at pdf_path using pdfplumber and ensures it's automatically closed after processing
    extracted_text = ""
    
    for page in pdf.pages:
        text = page.extract_text()
        if text:
            extracted_text += text
            
with open(output_text, "w", encoding = "utf-8") as f:        #Creates a text file in write mode with the UTF-8 encoding and closes it after writing
    f.write(extracted_text)
    
print(f"Text has been extracted from: {pdf_name}")
print(f"Total characters in the file are: {len(extracted_text)}")

Text has been extracted from: Machine Learning in Artificial Intelligence
Total characters in the file are: 40503


In [4]:
print("=" *50)       #Prints a horizontal divider of 50 equal signs to visually seperate sections in console output
print ("PREVIEW (first 1000 characters)")
print("=" *50)
print(extracted_text[:1000])
print("=" *50)

PREVIEW (first 1000 characters)
Proceedingsofthe52ndHawaiiInternationalConferenceonSystemSciences|2019
Machine Learning in Artificial Intelligence:
Towards a Common Understanding
Niklas Kühl Marc Goutier Robin Hirt Gerhard Satzger
Karlsruhe Institute of Karlsruhe Institute of Karlsruhe Institute of Karlsruhe Institute of
Technology Technology Technology Technology
kuehl@kit.edu marc.goutier@kit.edu hirt@kit.edu gerhard.satzger@kit.edu
Abstract machine learning within instantiations of artificial
intelligence, precisely within intelligent agents. To do
The application of “machine learning” and “arti- so, we take a machine learning perspective on the
ficial intelligence” has become popular within the last capabilities of intelligent agents as well as the
decade. Both terms are frequently used in science and corresponding implementation.
media, sometimes interchangeably, sometimes with The contribution of our paper is threefold. First, we
different meanings. In this work, we aim to clarif

In [5]:
from transformers import pipeline

summarizer = pipeline("summarization", model = "sshleifer/distilbart-cnn-12-6") #Creates a hugging face transformer summarization pipeline using a pretrained CNN model

#Other summarizers to be considered can be "facebook/bart-large-cnn", "falconsai/text_summarization", "google/t5-small" 

summary = summarizer(extracted_text[:1500], max_length = 150, min_length = 50, do_sample = False)
summary_text = summary[0]['summary_text']

print("=" *50)
print("SUMMARY")
print("=" *50)
print(summary_text)
print("=" *50)

C:\Users\sarve\anaconda3\envs\docanalysis\lib\site-packages\transformers\utils\generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
W0222 21:32:01.358000 37528 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
C:\Users\sarve\anaconda3\envs\docanalysis\lib\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


SUMMARY
 The application of “machine learning” and “arti- so, we take a machine learning perspective on the last capabilities of intelligent agents as well as the last decade . Both terms are frequently used in science and corresponding implementation . Proceedingsofthe52ndHawaiiInternationalConferenceonSystemSciences|2019 .


In [6]:
import nltk
nltk.download('punkt', quiet = True)
nltk.download('punkt_tab', quiet = True)
from nltk.tokenize import sent_tokenize

sentences = sent_tokenize(extracted_text)

passages = []
current_passage = ""
for sentence in sentences:
    if len(current_passage.split()) + len(sentence.split()) < 250:    #Adjusts 250 word limit per passage if required
        current_passage += "" + sentence
        
    else:
        passages.append(current_passage.strip())
        current_passage = sentence
        
if current_passage:
    passages.append(current_passage.strip())
    
print(f"Document is split into {len(passages)} passages")

Document is split into 25 passages


In [7]:
#Generates questions and answers and then saves to file

qg_pipeline = pipeline("text2text-generation", model = "valhalla/t5-base-qg-hl")
qa_pipeline = pipeline("question-answering", model = "deepset/roberta-base-squad2")


def generate_questions(passage, min_questions = 2):
    input_text = f"generate questions: {passage}"
    result = qg_pipeline(input_text)
    questions = result[0]['generated_text']. split('<sep>')
    questions = [q.strip() for q in questions if q.strip()]
#We initialize Question generation and Q/A pipelines using a pretrained transformer and defines it in the function

    if len(questions) < min_questions:
        passage_sentences = passage.split('.')
        for i in range(len(passage_sentences)):
            if len(questions) >= min_questions:
                break
                
            additional_input = '.'.join(passage_sentences[i:i+2])
            additional_results = qg_pipeline(f"generate questions: {additional_input}")
            additional_questions = additional_results[0]['generated_text'].split('<sep>')
            questions.extend([q.strip() for q in additional_questions if q.strip()])
            
    return questions[:min_questions]
#We generate additional questions if there are fewer questions asked than required until the amount is reached

answered_questions = set()
seperator = "-" *50

with open(output_result, "w", encoding = "utf-8") as results_file:
    
    header = f"DOCUMENT ANALYSIS FILE\nFile: {pdf_name}\n{'-' *50}\n"
    header += f"\nSummary\n{seperator}\n{summary_text}\n{'-' *50}\n\nQ&A BY PASSAGE\n{'-' *50}\n"
    print (header)
    results_file.write(header)
    
    for idx, passage in enumerate(passages):
        passage_header = f"\nPASSAGE {idx +1}\n{seperator}\n{passage}\n{seperator}\n"
        print(passage_header)
        results_file.write(passage_header)
        
        questions = generate_questions(passage)
        
        for question in questions:
            if question not in answered_questions:
                answer = qa_pipeline({'question': question, 'context': passage})
                qa_block = f"Question: {question}\nAnswer: {answer['answer']}\n"
                print(qa_block)
                results_file.write(qa_block)
                answered_questions.add(question)
                
        results_file.write(f"{'=' *50}\n")
        print("=" *50)
#We are making a formatted document report and saving the summary by going through each passage and generate unique questions and extracting answers from the model

print(f"The results are saved to: {output_result}")

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thouroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


C:\Users\sarve\anaconda3\envs\docanalysis\lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sarve\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to see activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


DOCUMENT ANALYSIS FILE
File: Machine Learning in Artificial Intelligence
--------------------------------------------------

Summary
--------------------------------------------------
 The application of “machine learning” and “arti- so, we take a machine learning perspective on the last capabilities of intelligent agents as well as the last decade . Both terms are frequently used in science and corresponding implementation . Proceedingsofthe52ndHawaiiInternationalConferenceonSystemSciences|2019 .
--------------------------------------------------

Q&A BY PASSAGE
--------------------------------------------------


PASSAGE 1
--------------------------------------------------
Proceedingsofthe52ndHawaiiInternationalConferenceonSystemSciences|2019
Machine Learning in Artificial Intelligence:
Towards a Common Understanding
Niklas Kühl Marc Goutier Robin Hirt Gerhard Satzger
Karlsruhe Institute of Karlsruhe Institute of Karlsruhe Institute of Karlsruhe Institute of
Technology Technology Tec

C:\Users\sarve\anaconda3\envs\docanalysis\lib\site-packages\transformers\generation\utils.py:1273: UserWarning: Using the model-agnostic default `max_length` (=20) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Question: What is the main focus of the paper?
Answer: we seek to provide more gent agents

Question: What is the name of the conference?
Answer: ndHawaiiInternationalConferenceonSystemSciences


PASSAGE 2
--------------------------------------------------
Introduction
relevant literature in the fields of machine learning and
artificial intelligence.Next, we present and elaborate
In his US senate hearing in April 2018, Mark
our conceptual framework which highlights the con-
Zuckerberg stressed the necessary capabilities of
tribution of machine learning to artificial intelligence.Facebook’s “AI tools (…) to (…) identify hate speech
On that basis, we derive an agenda for future research
(…)” or “ (…) terrorist propaganda” [1].Researchers
and conclude with a summary, current limitations, as
would typically describe such tasks of identifying
well as an outlook.specific instances within social media platforms as
classification tasks within the field of (supervised)
2.Related work
machine le

Token indices sequence length is longer than the specified maximum sequence length for this model (596 > 512). Running this sequence through the model will result in indexing errors


Question: What is the executing backend of knowledge?
Answer: learning
backend


PASSAGE 18
--------------------------------------------------
Although at an early stage, our framework should
allow scientists and practitioners to be more precise
when referring to machine learning and AI.It
highlights the importance of not using the terms
Page5242References [13] O. Bousquet, U. von Luxburg, and G. Rätsch,
Advanced Lectures on Machine Learning: ML
Summer Schools 2003, Canberra, Australia,
[1] “The Washington Post,” “Transcript of Mark
February 2-14, 2003, Tübingen, Germany, August
Zuckerberg’s Senate hearing,” 2018.[Online].4-16, 2003, Revised Lectures, vol.3176.Springer,
Available:
2011.
https://www.washingtonpost.com/news/the-
switch/wp/2018/04/10/transcript-of-mark- [14] M. Mohri, A. Rostamizadeh, and A. Talwalkar,
zuckerbergs-senate- Foundations of machine learning.MIT press, 2012.
hearing/?utm_term=.4720e7f10b41.[Accessed:
15-Jun-2018].[15] G.-B.Huang, Q.-Y.Zhu, and C.-K. Siew, “Ext